# 🧭 Séance 3 — Mini‑projet Titanic : de Pandas à un tableau propre et son journal
_Master IA‑GI — Python pour les sciences de données_

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yelallioui/python-for-ds-s03/blob/main/mini-projet-03/mini-projet-03.ipynb)

**Comment ouvrir ce notebook :** sur votre ordinateur, ouvrez-le **depuis son dossier** `mini-projet-03/`, sans le déplacer. Les fichiers de départ sont dans `data/`, à côté : `titanic.csv` (le fichier brut) et `s2_titanic.csv` (celui que votre mini-projet de la séance 2 a produit, en version de référence, pour que tout le monde parte du même point) ; **vos résultats** sont écrits dans `result/`, à côté aussi. Pandas doit être installé (`pip install pandas` ; déjà présent avec Anaconda).

**Dans Colab**, le notebook arrive **seul** : exécutez la cellule **Étape 0** (plus bas) pour récupérer le dossier du TP, avec ses deux fichiers dans `data/`. Pour **garder votre travail**, faites **Fichier ▸ Enregistrer une copie dans Drive** (ou dans GitHub) : Colab ne sauvegarde pas un notebook ouvert depuis GitHub.

**Objectifs** — à la fin de ce notebook, vous saurez :
- **charger** deux fichiers CSV avec Pandas et les **réunir** en un seul DataFrame, face à leur numéro de passager
- **regarder**, **sélectionner** et **auditer** le tableau : valeurs manquantes, doublons, tarifs nuls
- **décider** quoi faire des trous : supprimer, remplir, marquer ou laisser, en **annonçant le coût** de chaque choix
- **corriger** les types, **créer**, renommer et trier des colonnes, puis **mesurer l'effet** du nettoyage sur un résultat
- **écrire** deux nouveaux fichiers CSV : `s3_titanic_propre.csv` et son **journal** `s3_journal.csv`, le point de départ de la séance 4

**Durée estimée** : 45 minutes environ · **Parties** :
- **M** · Mini-projet : le Titanic avec Pandas

**Comment travailler seul**
1. Exécutez les cellules **dans l'ordre** (`Maj + Entrée`).
2. Lisez le *cours en bref*, puis **exécutez le code complet** : c'est lui qui contient les détails des diapos.
3. Faites chaque exercice en remplaçant les `...` par votre code, puis exécutez la cellule : un ✅ ou un ❌ s'affiche.
4. Un ❌ ? Ouvrez l'**indice 1**, puis 2, puis 3.

### 🧰 Pour commencer : importer, puis se contrôler
Exécutez la cellule ci-dessous. Elle importe ce qu'il faut et définit **`verifier`**, la fonction qui affiche ✅ ou ❌ sous chaque exercice. Rien de nouveau dans son écriture : une fonction (`def`), un `try` / `except` et un `lambda`, vus en séance 1.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

print("Pandas", pd.__version__, "· NumPy", np.__version__)

def verifier(ex, description, test):
    """Affiche ✅ si test() est vrai, ❌ sinon (même si le test provoque une erreur)."""
    try:
        ok = bool(test())
    except Exception:
        ok = False
    print(("✅ " if ok else "❌ ") + ex + " · " + description)

### 🪪 Qui êtes-vous ?
Écrivez votre **nom** et votre **prénom** **entre guillemets**, en remplaçant les `...` (par exemple `nom = "Benali"`). Ces deux lignes me permettent de reconnaître votre travail : le contrôle reste ❌ tant qu'elles ne sont pas remplies. **Votre dépôt est public : n'y écrivez jamais de donnée personnelle (CNE, mot de passe, téléphone).**

In [ ]:
nom = ...
prenom = ...

print(nom, prenom)
verifier("identité", "nom et prénom renseignés", lambda: type(nom) is str and type(prenom) is str and len(nom.strip()) >= 2 and len(prenom.strip()) >= 2)

### 📁 Étape 0 · Récupérer le dossier du TP (dans Colab)
Dans **Colab**, le notebook arrive **seul** : les fichiers `titanic.csv` et `s2_titanic.csv` **restent sur GitHub**. Exécutez la cellule ci-dessous : elle **télécharge le dossier du TP** dans la session Colab, puis s'y place. Les fichiers se trouvent alors dans `python-for-ds-s03/mini-projet-03/data/` (panneau **📁** à gauche, bouton **↻** pour actualiser).

**Sur votre ordinateur**, cette cellule ne fait rien : ouvrez simplement le notebook depuis son dossier.

Les fichiers d'une session Colab sont **temporaires** : ils disparaissent à la fermeture. Pour garder un fichier, par exemple `result/s3_titanic_propre.csv` : clic droit dessus ▸ **Télécharger**.

In [ ]:
import os
import subprocess

try:
    import google.colab                 # ce module n'existe que dans Colab
    dans_colab = True
except ImportError:
    dans_colab = False

if dans_colab and not os.path.exists("data/titanic.csv"):
    subprocess.run(["git", "clone", "-q", "https://github.com/yelallioui/python-for-ds-s03"])
    os.chdir("python-for-ds-s03/mini-projet-03")

print("dossier de travail :", os.getcwd())
print("data/titanic.csv trouvé :", os.path.exists("data/titanic.csv"))

**Le fil de data science continue.** À la séance 2, NumPy vous a fait produire `s2_titanic.csv` : les **sept colonnes numériques** du Titanic et **quatre colonnes calculées**. Mais NumPy s'arrêtait là : le sexe, le nom, la cabine et le port n'y entraient pas. **Pandas lève cet obstacle** : il lit tout le fichier, texte compris, avec des noms de colonnes et des types par colonne.

**La question qui traverse les six séances : qui a survécu au naufrage du Titanic, et pourquoi ?** Avant de répondre, il faut un tableau **propre** : aujourd'hui, vous **chargez** les données, vous les **auditez**, vous **décidez** quoi faire de leurs défauts, et vous **notez chaque décision** dans un journal. Vous produisez deux fichiers CSV : `s3_titanic_propre.csv` et `s3_journal.csv`. Chaque étape se **contrôle toute seule** : un ✅ ou un ❌ vous dit si c'est juste.

**Le fil du mini‑projet**
- **Charger** : **M1** lire les deux fichiers et les réunir · **M2** le portrait du tableau
- **Interroger** : **M3** sélectionner colonnes, lignes et masques
- **Auditer** : **M4** les six gestes de diagnostic · **M5** les trous ne tombent pas au hasard
- **Décider** : **M6** le coût de supprimer, puis marquer, remplir ou laisser · **M7** libellés et types
- **Fabriquer** : **M8** renommer, créer et trier des colonnes
- **Juger** : **M9** deux nettoyages, deux résultats
- **Résultat** : **M10** le journal, puis `result/s3_titanic_propre.csv` et `result/s3_journal.csv`
- **Fichiers de départ** : `data/titanic.csv` et `data/s2_titanic.csv`, à côté de ce notebook : **c'est à vous de les charger** (M1). _Aucun geste nouveau : tout vient de la partie 1 et du cours._

**🧭 Étape M1 — Charger les deux fichiers et les réunir**

**Comment charger `data/s2_titanic.csv` (la séance 2) et `data/titanic.csv` (le fichier brut), puis les réunir en un seul DataFrame `df` ?** Le numéro de passager, `PassengerId`, relie les deux fichiers : mettez-le en **index** des deux. Le fichier de la séance 2 a 11 colonnes numériques ; le fichier brut a les **cinq colonnes de texte** qui lui manquaient : `Name`, `Sex`, `Ticket`, `Cabin` et `Embarked`.

Vous écrivez trois choses : `s2` (le premier fichier), `brut` (le second) et `df` : **le contenu de `s2`, plus les cinq colonnes de texte prises dans `brut`**. Au final : **891 lignes** et **15 colonnes** (`PassengerId` est l'index).

<details>
<summary>💡 Indice 1 · la piste</summary>

Pour chaque fichier : `pd.read_csv(chemin, index_col=...)`. Les deux fichiers ont en commun les colonnes numériques : vous n'avez besoin que des cinq colonnes de texte du second.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

Partez d'une **copie** de `s2` (`s2.copy()`), puis ajoutez une colonne de texte à la fois, prise dans `brut` : l'**alignement par étiquette** (partie A, exercice A5) range chaque valeur face à son numéro de passager.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`s2 = pd.read_csv("data/s2_titanic.csv", index_col="PassengerId")` ; `brut = pd.read_csv("data/titanic.csv", index_col="PassengerId")` ; `df = s2.copy()` ; puis `for colonne in ["Name", "Sex", "Ticket", "Cabin", "Embarked"]:` et, dans la boucle, `df[colonne] = brut[colonne]`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
# votre code : lisez les deux fichiers de data/, puis rassemblez-les dans df
s2 = ...
brut = ...
df = ...

verifier("M1", "df : 891 passagers, 15 colonnes, numéro de passager en index", lambda: isinstance(df, pd.DataFrame) and df.shape == (891, 15) and df.index.name == "PassengerId")
verifier("M1", "les 10 colonnes de la séance 2, puis les 5 colonnes de texte", lambda: list(df.columns)[:10] == ["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare", "taille_famille", "seul", "fare_par_personne", "log_fare"] and list(df.columns)[10:] == ["Name", "Sex", "Ticket", "Cabin", "Embarked"])
verifier("M1", "chaque texte est face au bon passager : le numéro 1 est Braund, Mr. Owen Harris, un homme", lambda: df.loc[1, "Name"] == "Braund, Mr. Owen Harris" and df.loc[1, "Sex"] == "male")
verifier("M1", "les trous sont là : 177 âges, 687 cabines, 2 ports", lambda: df["Age"].isna().sum() == 177 and df["Cabin"].isna().sum() == 687 and df["Embarked"].isna().sum() == 2)

_À partir d'ici, la cellule ci‑dessous refait ce que vous venez d'écrire (`s2`, `brut` et `df`) : les étapes suivantes repartent d'une base correcte, même si l'étape M1 vous a résisté._

In [ ]:
s2 = pd.read_csv("data/s2_titanic.csv", index_col="PassengerId")
brut = pd.read_csv("data/titanic.csv", index_col="PassengerId")
df = s2.copy()
for colonne in ["Name", "Sex", "Ticket", "Cabin", "Embarked"]:
    df[colonne] = brut[colonne]
print(df.shape)

**Regardez le tableau** avant de l'interroger : c'est le geste n° 1 de tout diagnostic. Exécutez la cellule ci‑dessous, puis lisez `info()` : les valeurs **présentes** de chaque colonne, et son **type**.

In [ ]:
print(df.head(3))
df.info()

**🧭 Étape M2 — Le portrait du tableau**

Sans lire la sortie à l'œil, **calculez** :
1. `nb_lignes` et `nb_colonnes` : la forme de `df`
2. `nb_ages` : le nombre d'**âges présents**
3. `age_moyen` : la moyenne des âges (les trous sont ignorés)
4. `effectifs_sexe` : le nombre de passagers **par sexe** (`value_counts`)

<details>
<summary>💡 Indice 1 · la piste</summary>

`shape`, `count()`, `mean()` et `value_counts()` : les gestes de la partie B.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`nb_lignes, nb_colonnes = df.shape` ; `df["Age"].count()` ; `df["Age"].mean()` ; `df["Sex"].value_counts()`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`nb_ages = df["Age"].count()` vaut 714 (891 moins 177 trous) ; `effectifs_sexe = df["Sex"].value_counts()`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
nb_lignes, nb_colonnes = ..., ...
nb_ages = ...
age_moyen = ...
effectifs_sexe = ...

verifier("M2", "891 passagers, 15 colonnes", lambda: nb_lignes == 891 and nb_colonnes == 15)
verifier("M2", "714 âges présents, de moyenne 29,70", lambda: nb_ages == 714 and abs(age_moyen - 29.699) < 0.01)
verifier("M2", "577 hommes et 314 femmes", lambda: effectifs_sexe["male"] == 577 and effectifs_sexe["female"] == 314)
verifier("M2", "cohérence avec la séance 2 : 342 survivants et 537 passagers seuls, sur 891 lignes", lambda: nb_lignes == 891 and df["Survived"].sum() == 342 and df["seul"].sum() == 537)

**🧭 Étape M3 — Sélectionner : colonnes, lignes et masques**

Avec `df` (l'index est le numéro de passager, de 1 à 891 : **étiquette et position ne coïncident pas**) :
1. `enfants` : le sous-tableau des passagers de **moins de 12 ans**
2. `survie_enfants` : leur **taux de survie** (`Survived` vaut 1 ou 0 : sa moyenne est une proportion)
3. `femmes_1ere_30` : le sous-tableau des **femmes de 1re classe de moins de 30 ans**
4. `age_loc` et `age_iloc` : l'âge du passager numéro **5**, lu **par étiquette** puis **par position**

<details>
<summary>💡 Indice 1 · la piste</summary>

Un masque pour les lignes (`df[condition]`), une colonne et `mean()` pour un taux ; chaque condition entre parenthèses quand on les combine avec `&`. `loc` prend une étiquette, `iloc` une position.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`df[df["Age"] < 12]` ; `enfants["Survived"].mean()` ; `df[(df["Sex"] == "female") & (df["Pclass"] == 1) & (df["Age"] < 30)]` ; `df.loc[5, "Age"]` ; `df.iloc[4, df.columns.get_loc("Age")]`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`enfants` compte 68 passagers (un NaN compare faux : les âges inconnus ne sont pas dedans) ; `survie_enfants` vaut 0,574 ; `femmes_1ere_30` compte 30 passagères ; les deux âges valent 35.0.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
enfants = ...
survie_enfants = ...
femmes_1ere_30 = ...
age_loc = ...
age_iloc = ...

verifier("M3", "68 enfants de moins de 12 ans", lambda: isinstance(enfants, pd.DataFrame) and len(enfants) == 68 and (enfants["Age"] < 12).all())
verifier("M3", "57 % des enfants ont survécu (contre 38 % pour l'ensemble)", lambda: abs(survie_enfants - 0.5735) < 0.001)
verifier("M3", "30 femmes de 1re classe de moins de 30 ans", lambda: isinstance(femmes_1ere_30, pd.DataFrame) and len(femmes_1ere_30) == 30)
verifier("M3", "le passager 5 a 35 ans, lu par étiquette comme par position", lambda: age_loc == 35.0 and age_iloc == 35.0)

**🧭 Étape M4 — Auditer : les six gestes de diagnostic**

Avec `df`, faites l'**audit chiffré** du fichier :
1. `trous` : le nombre de valeurs manquantes **par colonne** (une Series), puis `colonnes_incompletes` : la **liste** des colonnes qui ont au moins un trou
2. `nb_doublons` : le nombre de **lignes en double**
3. `tarifs_nuls` : le nombre de passagers dont le tarif `Fare` vaut **0** (un billet gratuit : suspect)
4. `part_cabin` : la **part** de trous dans `Cabin` (un nombre entre 0 et 1)

<details>
<summary>💡 Indice 1 · la piste</summary>

Les gestes de la partie D : `isna()` marque, `sum()` compte, `mean()` donne une part, `duplicated()` repère les doubles.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`trous = df.isna().sum()` ; `list(trous[trous > 0].index)` ; `df.duplicated().sum()` ; `(df["Fare"] == 0).sum()` ; `df["Cabin"].isna().mean()`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

Trois colonnes incomplètes : `Age` (177), `Cabin` (687) et `Embarked` (2). Aucun doublon ; 15 tarifs nuls ; `Cabin` manque à 77 %.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
trous = ...
colonnes_incompletes = ...
nb_doublons = ...
tarifs_nuls = ...
part_cabin = ...

verifier("M4", "trous : une valeur par colonne, 177 âges, 687 cabines, 2 ports", lambda: isinstance(trous, pd.Series) and len(trous) == 15 and trous["Age"] == 177 and trous["Cabin"] == 687 and trous["Embarked"] == 2)
verifier("M4", "les colonnes incomplètes : Age, Cabin, Embarked", lambda: list(colonnes_incompletes) == ["Age", "Cabin", "Embarked"])
verifier("M4", "aucun doublon, 15 tarifs nuls", lambda: nb_doublons == 0 and tarifs_nuls == 15)
verifier("M4", "Cabin manque à 77 %", lambda: abs(part_cabin - 0.771) < 0.001)

**🧭 Étape M5 — Les trous ne tombent pas au hasard**

Si les âges manquaient **au hasard**, la part de trous serait voisine dans les trois classes. Mesurez-la :
1. `taux_age_manquant` : la **liste** de la part d'âges manquants en **1re, 2e et 3e classe** (dans cet ordre)
2. `survie_sans_age` : le taux de survie des passagers **sans âge** ; `survie_avec_age` : celui des passagers **dont l'âge est connu**
3. `constat` : en **une phrase** (texte), ce que cela change quand on **supprime** les lignes sans âge

<details>
<summary>💡 Indice 1 · la piste</summary>

Pour chaque classe : un masque `df["Pclass"] == classe`, la colonne `Age`, `isna()`, puis `mean()`. Une compréhension (séance 1) évite de répéter trois fois le même calcul.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`[df[df["Pclass"] == c]["Age"].isna().mean() for c in (1, 2, 3)]` ; `df[df["Age"].isna()]["Survived"].mean()` ; `df[df["Age"].notna()]["Survived"].mean()`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

Les parts valent environ 0,14 ; 0,06 et 0,28 : **les trous se concentrent en 3ᵉ classe**. Les passagers sans âge ont survécu à 29 %, les autres à 41 %. `constat` est un texte libre d'au moins 30 caractères.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
taux_age_manquant = ...
survie_sans_age = ...
survie_avec_age = ...
constat = ...

verifier("M5", "trois parts : 14 % en 1re classe, 6 % en 2e, 28 % en 3e", lambda: isinstance(taux_age_manquant, list) and len(taux_age_manquant) == 3 and abs(taux_age_manquant[0] - 0.1389) < 0.001 and abs(taux_age_manquant[1] - 0.0598) < 0.001 and abs(taux_age_manquant[2] - 0.277) < 0.001)
verifier("M5", "survie : 29 % sans âge, 41 % avec âge", lambda: abs(survie_sans_age - 0.2938) < 0.001 and abs(survie_avec_age - 0.4062) < 0.001)
verifier("M5", "constat rédigé (au moins 30 caractères)", lambda: isinstance(constat, str) and len(constat.strip()) >= 30)

**🧭 Étape M6 — Décider : supprimer, remplir, marquer ou laisser**

**Annoncez d'abord le coût**, puis décidez, **sans supprimer aucune ligne** de `df` :
1. `n_sans_trou` : le nombre de lignes qui resteraient avec `dropna()` ; `n_sans_age` : avec `dropna(subset=["Age"])`
2. `propre` : une **copie** de `df` où `Age` est **marqué puis rempli** par la médiane (colonne **`Age_estime`**), où les 2 ports `Embarked` manquants sont remplis par le **mode** (`mode()[0]`), et où `Cabin` est **laissée** et remplacée par une colonne booléenne **`A_cabine`** (`notna()`)

<details>
<summary>💡 Indice 1 · la piste</summary>

Dans cet ordre, sur la copie : marquer `Age_estime`, remplir `Age`, remplir `Embarked`, créer `A_cabine`. Marquer **avant** de remplir, sinon on ne sait plus où étaient les trous.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`propre = df.copy()` ; `propre["Age_estime"] = propre["Age"].isna()` ; `propre["Age"] = propre["Age"].fillna(propre["Age"].median())` ; `propre["Embarked"] = propre["Embarked"].fillna(propre["Embarked"].mode()[0])` ; `propre["A_cabine"] = propre["Cabin"].notna()`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`dropna()` laisserait 183 lignes sur 891, `dropna(subset=["Age"])` 714. La médiane des âges vaut 28 ; le mode du port est `S` ; 204 passagers ont une cabine.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
n_sans_trou = ...
n_sans_age = ...

propre = df.copy()
# votre code : marquer Age_estime, remplir Age par la médiane, remplir Embarked par le mode, créer A_cabine
...

verifier("M6", "le coût : 183 lignes avec dropna(), 714 avec dropna(subset=['Age'])", lambda: n_sans_trou == 183 and n_sans_age == 714)
verifier("M6", "propre : 891 lignes (rien supprimé), deux colonnes ajoutées", lambda: isinstance(propre, pd.DataFrame) and len(propre) == 891 and {"Age_estime", "A_cabine"} <= set(propre.columns))
verifier("M6", "Age : aucun trou, 177 âges marqués estimés, tous égaux à la médiane (28)", lambda: propre["Age"].isna().sum() == 0 and propre["Age_estime"].sum() == 177 and (propre.loc[propre["Age_estime"], "Age"] == 28.0).all())
verifier("M6", "Embarked : aucun trou, les 2 ports remplis par S", lambda: propre["Embarked"].isna().sum() == 0 and (propre.loc[df["Embarked"].isna(), "Embarked"] == "S").all())
verifier("M6", "Cabin laissée (687 trous) et A_cabine créée (204 vrais) ; df est intact", lambda: propre["Cabin"].isna().sum() == 687 and propre["A_cabine"].sum() == 204 and df["Age"].isna().sum() == 177)

**🧭 Étape M7 — Libellés et types**

Sur `propre` :
1. `effectifs_ports` : le nombre de passagers **par port** (`value_counts` sur `Embarked`)
2. `nb_sexes` : le nombre de **libellés distincts** de `Sex` une fois les espaces retirés et les minuscules appliquées (`.str.strip().str.lower()`) : y a-t-il des libellés incohérents ?
3. Convertissez `Pclass`, `Sex` et `Embarked` en type **`category`** (dans `propre`)

<details>
<summary>💡 Indice 1 · la piste</summary>

`value_counts()` révèle les libellés incohérents ; `.str.strip().str.lower()` les uniformise ; `astype("category")` range chaque libellé une fois.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`propre["Embarked"].value_counts()` ; `propre["Sex"].str.strip().str.lower().nunique()` ; `propre["Pclass"] = propre["Pclass"].astype("category")`, et de même pour `Sex` et `Embarked`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

Les ports : `S` 646, `C` 168, `Q` 77. `Sex` n'a que deux libellés : **rien à corriger**, et c'est aussi un résultat d'audit.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
effectifs_ports = ...
nb_sexes = ...
# votre code : convertissez Pclass, Sex et Embarked en category
...

verifier("M7", "ports : S 646, C 168, Q 77", lambda: effectifs_ports["S"] == 646 and effectifs_ports["C"] == 168 and effectifs_ports["Q"] == 77)
verifier("M7", "deux libellés pour le sexe : aucun libellé incohérent", lambda: nb_sexes == 2)
verifier("M7", "Pclass, Sex et Embarked sont de type category", lambda: all(str(propre[c].dtype) == "category" for c in ["Pclass", "Sex", "Embarked"]))
verifier("M7", "les valeurs n'ont pas changé : la somme des classes vaut toujours 2057", lambda: str(propre["Pclass"].dtype) == "category" and propre["Pclass"].astype(int).sum() == 2057)

**🧭 Étape M8 — Renommer, créer, trier**

Sur `propre` :
1. **Renommez** les quatre colonnes calculées en séance 2 pour qu'elles suivent la même écriture que les autres : `taille_famille` → `FamilySize`, `seul` → `Seul`, `fare_par_personne` → `Fare_par_personne` et `log_fare` → `Log_fare`
2. **Créez** la colonne booléenne `Famille_nombreuse` : vrai quand `FamilySize` vaut **5 ou plus**
3. `classement` : une **copie triée** de `propre` par `Fare`, du **plus grand au plus petit** ; `propre` reste dans l'ordre des numéros de passager

<details>
<summary>💡 Indice 1 · la piste</summary>

`rename(columns={...})` prend un dictionnaire ancien nom → nouveau nom et rend un **nouveau** DataFrame ; `sort_values` aussi : il faut garder le résultat.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`propre = propre.rename(columns={"taille_famille": "FamilySize", ...})` ; `propre["Famille_nombreuse"] = propre["FamilySize"] >= 5` ; `classement = propre.sort_values("Fare", ascending=False)`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

62 passagers voyagent en famille nombreuse. Les trois plus gros tarifs (512,33) sont ceux des passagers 259, 680 et 738.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
# votre code : renommez les quatre colonnes, créez Famille_nombreuse
...

classement = ...

verifier("M8", "les quatre colonnes sont renommées", lambda: {"FamilySize", "Seul", "Fare_par_personne", "Log_fare"} <= set(propre.columns) and not ({"taille_famille", "seul", "fare_par_personne", "log_fare"} & set(propre.columns)))
verifier("M8", "Famille_nombreuse : 62 passagers", lambda: "Famille_nombreuse" in propre.columns and propre["Famille_nombreuse"].sum() == 62)
verifier("M8", "classement : les passagers 259, 680 et 738 (tarif 512,33) en tête", lambda: isinstance(classement, pd.DataFrame) and len(classement) == 891 and set(classement.index[:3]) == {259, 680, 738})
verifier("M8", "propre reste dans l'ordre des numéros de passager", lambda: "Seul" in propre.columns and propre.index[0] == 1 and propre.index[-1] == 891)

_Les deux dernières étapes repartent d'un tableau correct : la cellule ci‑dessous reconstruit `propre` tel que M6, M7 et M8 le demandent, que vous ayez réussi ou non. **`df`, lui, reste le tableau brut**, tel qu'il a été chargé._

In [ ]:
propre = df.copy()
propre["Age_estime"] = propre["Age"].isna()
propre["Age"] = propre["Age"].fillna(propre["Age"].median())
propre["Embarked"] = propre["Embarked"].fillna(propre["Embarked"].mode()[0])
propre["A_cabine"] = propre["Cabin"].notna()
for colonne in ["Pclass", "Sex", "Embarked"]:
    propre[colonne] = propre[colonne].astype("category")
propre = propre.rename(columns={"taille_famille": "FamilySize", "seul": "Seul", "fare_par_personne": "Fare_par_personne", "log_fare": "Log_fare"})
propre["Famille_nombreuse"] = propre["FamilySize"] >= 5
print(propre.shape, "· trous dans Age :", propre["Age"].isna().sum())

**🧭 Étape M9 — Deux nettoyages, deux résultats**

Mesurez la **survie des passagers de 3ᵉ classe** (la moyenne de `Survived`) avec **trois tableaux** : `df` complet, `df.dropna()` et `df.dropna(subset=["Age"])`. **Utilisez `df`** (le tableau brut), pas `propre`.
1. `survie_brute`, `survie_dropna` et `survie_age` : les trois taux
2. `n_3e_dropna` : le nombre de passagers de 3ᵉ classe qui restent avec `dropna()`
3. `conclusion` : en **une phrase** (texte), ce que montre l'écart entre les trois taux

<details>
<summary>💡 Indice 1 · la piste</summary>

Pour chaque tableau : un masque `["Pclass"] == 3`, la colonne `Survived`, `mean()`. Écrivez-le une fois pour `df`, puis changez seulement le tableau de départ.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`df[df["Pclass"] == 3]["Survived"].mean()` ; `df.dropna()[df.dropna()["Pclass"] == 3]["Survived"].mean()` ; et `n_3e_dropna = (df.dropna()["Pclass"] == 3).sum()`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

Environ 0,24 pour `df` ; **0,50** avec `dropna()`, calculé sur **10 passagers** seulement ; 0,24 avec `dropna(subset=["Age"])`. Aucun n'est faux : ils ne répondent pas à la même question.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
survie_brute = ...
survie_dropna = ...
survie_age = ...
n_3e_dropna = ...
conclusion = ...

verifier("M9", "survie en 3e classe : 24 % pour df, 24 % avec dropna(subset=['Age'])", lambda: abs(survie_brute - 0.2424) < 0.001 and abs(survie_age - 0.2394) < 0.001)
verifier("M9", "survie en 3e classe : 50 % avec dropna()", lambda: abs(survie_dropna - 0.5) < 0.001)
verifier("M9", "dropna() ne laisse que 10 passagers de 3e classe", lambda: n_3e_dropna == 10)
verifier("M9", "conclusion rédigée (au moins 30 caractères)", lambda: isinstance(conclusion, str) and len(conclusion.strip()) >= 30)

**🧭 Étape M10 — Le résultat de la séance : un tableau propre et son journal**

Écrivez **`s3_titanic_propre.csv`** et **`s3_journal.csv`** dans le dossier `result/` :
1. `journal` : un DataFrame d'**au moins 5 lignes**, **une par décision**, avec les colonnes `Colonne`, `Decision`, `Pourquoi` et `Lignes_touchees` (le nombre de lignes concernées)
2. Écrivez `propre` dans `chemin_propre` (**avec** son index, le numéro de passager) et `journal` dans `chemin_journal` (**sans** index), avec `to_csv`

La séance 4 (Pandas, regrouper et croiser) reprendra `s3_titanic_propre.csv`.

<details>
<summary>💡 Indice 1 · la piste</summary>

Le journal est un DataFrame construit depuis une liste de dictionnaires, un dictionnaire par décision. Les décisions : `Age`, `Embarked`, `Cabin`, les types, les colonnes renommées.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`pd.DataFrame([{"Colonne": "Age", "Decision": "remplie par la médiane (28), marquée Age_estime", "Pourquoi": "20 % de trous", "Lignes_touchees": 177}, ...])` ; puis `propre.to_csv(chemin_propre)` et `journal.to_csv(chemin_journal, index=False)`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

Cinq décisions : `Age` (177 lignes), `Embarked` (2), `Cabin` (0 : laissée, `A_cabine` ajoutée), les types (891) et les colonnes renommées (891).

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
if Path("data").exists():                                                  # le notebook est bien ouvert depuis son dossier
    Path("result").mkdir(exist_ok=True)                                      # le dossier result, créé s'il manque
chemin_propre = Path("result") / "s3_titanic_propre.csv"
chemin_journal = Path("result") / "s3_journal.csv"

journal = ...
# votre code : écrivez propre dans chemin_propre et journal dans chemin_journal avec to_csv

verifier("M10", "s3_titanic_propre.csv : 891 lignes, 18 colonnes, le numéro de passager en première colonne", lambda: pd.read_csv(chemin_propre, index_col="PassengerId").shape == (891, 18))
verifier("M10", "Age sans trou, 177 âges marqués estimés ; Embarked sans trou", lambda: pd.read_csv(chemin_propre)["Age"].isna().sum() == 0 and pd.read_csv(chemin_propre)["Age_estime"].sum() == 177 and pd.read_csv(chemin_propre)["Embarked"].isna().sum() == 0)
verifier("M10", "Cabin laissée (687 trous), A_cabine : 204 vrais, Famille_nombreuse : 62 vrais", lambda: pd.read_csv(chemin_propre)["Cabin"].isna().sum() == 687 and pd.read_csv(chemin_propre)["A_cabine"].sum() == 204 and pd.read_csv(chemin_propre)["Famille_nombreuse"].sum() == 62)
verifier("M10", "le passager 1 : 22 ans (non estimé), tarif 7,25, FamilySize 2", lambda: pd.read_csv(chemin_propre).iloc[0]["Age"] == 22.0 and pd.read_csv(chemin_propre).iloc[0]["Fare"] == 7.25 and pd.read_csv(chemin_propre).iloc[0]["FamilySize"] == 2)
verifier("M10", "s3_journal.csv : au moins 5 décisions et les quatre colonnes demandées", lambda: len(pd.read_csv(chemin_journal)) >= 5 and {"Colonne", "Decision", "Pourquoi", "Lignes_touchees"} <= set(pd.read_csv(chemin_journal).columns))

**🎉 Votre tableau propre et son journal sont prêts.** Vous avez `s3_titanic_propre.csv` : les **15 colonnes** du Titanic réunies, **les trous traités par décision** (rien n'a été supprimé), les types corrigés et trois colonnes ajoutées (`Age_estime`, `A_cabine`, `Famille_nombreuse`) ; et `s3_journal.csv`, qui dit **ce que vous avez décidé, pourquoi, et ce que cela touche**. Ils sont dans le dossier `result/` de votre mini‑projet.

**Une question de réflexion, sans code :** quelle analyse votre nettoyage **rend‑il impossible** ? _(Par exemple : étudier le numéro de pont, puisque `Cabin` n'a pas été exploitée, ou comparer finement les âges entre classes, puisque la médiane les rapproche.)_

---
## 🚧 Ce que vous ne savez pas encore faire
Votre tableau est propre, mais quatre questions du Titanic restent **hors de portée avec ces seuls gestes** : ce sont celles de la séance 4.

1. **Quel est le taux de survie de chaque classe, de chaque port, de chaque classe dans chaque port ?** Il faudrait répéter un masque et une moyenne pour chaque groupe : le **regroupement** (`groupby`) le fait en une opération.
2. **Quel est l'âge médian par classe et par titre (Mr, Mrs, Miss…) ?** Le titre est caché dans le texte de la colonne `Name`, et la médiane globale que vous avez utilisée pour remplir les âges est **la même pour tous**.
3. **Dans quelle ville et à quelle date chaque passager a-t-il embarqué ?** Il faudrait **fusionner** `df` avec une petite table des escales du Titanic.
4. **Combien de passagers dans chaque tranche d'âge ?** Il faudrait découper `Age` en tranches.

**À la séance 4**, je vous remettrai le dossier `seance-04`, dont le mini‑projet contient **ce fichier** : `s3_titanic_propre.csv`. Pandas y **regroupera, croisera, fusionnera et datera** les passagers.